# Celda 1: procesos por modalidad

In [0]:
%sql
USE CATALOG workspace_practica_1;

CREATE OR REPLACE TABLE secop_gold.procesos_por_modalidad AS
SELECT modalidad,
       COUNT(*) AS num_procesos,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_procesos,
       SUM(precio_base) AS precio_base_total,
       ROUND(100 * SUM(precio_base) / SUM(SUM(precio_base)) OVER (), 2) AS pct_valor
FROM secop_silver.procesos
GROUP BY modalidad
ORDER BY precio_base_total DESC;

SELECT * FROM secop_gold.procesos_por_modalidad;

# Celda 2: competencia (cuántas ofertas recibe cada proceso)

In [0]:
%sql
CREATE OR REPLACE TABLE secop_gold.competencia_por_modalidad AS
SELECT modalidad,
       COUNT(*) AS num_procesos,
       ROUND(AVG(num_respuestas), 2) AS promedio_ofertas,
       ROUND(100 * AVG(CASE WHEN num_respuestas = 1 THEN 1 ELSE 0 END), 2) AS pct_oferente_unico,
       ROUND(100 * AVG(CASE WHEN num_respuestas = 0 THEN 1 ELSE 0 END), 2) AS pct_sin_ofertas
FROM secop_silver.procesos
GROUP BY modalidad
ORDER BY num_procesos DESC;

SELECT * FROM secop_gold.competencia_por_modalidad;

# Celda 3: diferencia entre precio base y valor adjudicado

In [0]:
%sql
CREATE OR REPLACE TABLE secop_gold.ahorro_por_modalidad AS
SELECT modalidad,
       COUNT(*) AS procesos_adjudicados,
       SUM(precio_base) AS precio_base_total,
       SUM(valor_adjudicado) AS valor_adjudicado_total,
       ROUND(100 * (1 - SUM(valor_adjudicado) / SUM(precio_base)), 2) AS pct_ahorro
FROM secop_silver.procesos
WHERE valor_adjudicado > 0
  AND NOT adjudicacion_multiple
GROUP BY modalidad
ORDER BY procesos_adjudicados DESC;

SELECT * FROM secop_gold.ahorro_por_modalidad;

# Celda 4: evolución mensual

In [0]:
%sql
CREATE OR REPLACE TABLE secop_gold.procesos_mensual AS
SELECT date_trunc('month', fecha_publicacion) AS mes,
       COUNT(*) AS num_procesos,
       SUM(precio_base) AS precio_base_total
FROM secop_silver.procesos
GROUP BY 1
ORDER BY 1;

SELECT * FROM secop_gold.procesos_mensual;

# Celda 5: top entidades y top proveedores

In [0]:
%sql
CREATE OR REPLACE TABLE secop_gold.top_entidades AS
SELECT entidad, COUNT(*) AS num_procesos, SUM(precio_base) AS precio_base_total
FROM secop_silver.procesos
GROUP BY entidad
ORDER BY precio_base_total DESC
LIMIT 15;

CREATE OR REPLACE TABLE secop_gold.top_proveedores AS
SELECT proveedor, COUNT(DISTINCT id_del_proceso) AS procesos_ganados
FROM secop_silver.adjudicaciones
GROUP BY proveedor
ORDER BY procesos_ganados DESC
LIMIT 15;

SELECT * FROM secop_gold.top_proveedores;

Hallazgos
- El 94 % de los procesos no compite. Contratación directa (57,8 %) y régimen especial (36,2 %) concentran casi todos los procesos y el 65 % del valor. Por eso salen con 100 % “sin ofertas”: esas modalidades no reciben ofertas por diseño.
- Mínima cuantía es donde más se ahorra (10,9 %) y donde hay más competencia (3,15 ofertas en promedio). En las licitaciones públicas el ahorro es casi nulo (0,13 %).
- Contratación directa “con ofertas”: 82,5 % tiene un único oferente. Es el dato más llamativo sobre competencia.
- Febrero es el pico del año: 4.083 procesos y el 22 % del valor total. Enero y febrero concentran la contratación de inicio de año; diciembre cae a la mitad.
- Un ahorro negativo no siempre es malo. En enajenación de bienes (-33,5 %) la entidad vende, así que recibir más del precio base es una ganancia.


-----------------------------------------